Imports


In [1]:
import os
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding
import numpy as np
import glob

2025-11-19 17:53:20.631132: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-19 17:53:21.309797: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-19 17:53:23.808473: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Get text and create sequences

In [2]:
path_text = tf.keras.utils.get_file('shakespeare.txt', 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt')
text = open(path_text, 'rb').read().decode(encoding='utf-8')
chars = sorted(set(text))
vocab_size = len(chars)
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = np.array(chars)

text_as_int = np.array([char2idx[c] for c in text], dtype=np.int32)

seq_length = 100
examples_per_epoch = len(text_as_int) // (seq_length + 1)

char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

2025-11-19 17:53:28.965349: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Create splits based on sequence, then prepare for training using batch & buffer size

In [3]:
def split_input_target(chunk):
    input_seq = chunk[:-1]
    target_char = chunk[-1]
    return input_seq, target_char

dataset = sequences.map(split_input_target)

BATCH_SIZE = 64
BUFFER_SIZE = 10000

dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)

Create model

In [4]:
embedding_dim = 256
lstm_units = 512

model = Sequential([
    Embedding(vocab_size, embedding_dim),
    LSTM(lstm_units, return_sequences=True),
    LSTM(lstm_units),
    Dense(vocab_size, activation='softmax')
])

Train model, store weights as checkpoint file

In [5]:
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

checkpoint_dir = './training_checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch:02d}.weights.h5")

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_prefix,
    save_weights_only=True,
    save_best_only=False,
    save_freq='epoch'
)


In [8]:
EPOCH = 1
model.fit(dataset, epochs=EPOCH, callbacks=[checkpoint_callback])

172/172 ━━━━━━━━━━━━━━━━━━━━ 227s 1s/step - accuracy: 0.1989 - loss: 3.0400


In [9]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (64, 100, 256)         │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (64, 100, 512)         │     1,574,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (64, 512)              │     2,099,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (64, 65)               │        33,345 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,172,293 (42.62 MB)

 Trainable params: 3,724,097 (14.21 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 7,448,196 (28.41 MB)

Generate text

In [10]:
def latest_checkpoint():
    weight_files = sorted(glob.glob(os.path.join(checkpoint_dir, "*.weights.h5")))
    return weight_files[-1] if weight_files else None
weights_to_load = latest_checkpoint()
model.load_weights(weights_to_load)

def generate_text(seed_text, num_generate=400, temperature=0.7):

    pad_idx = 0
    input_ids = [char2idx.get(c, pad_idx) for c in seed_text]
    if len(input_ids) < seq_length:
        input_ids = [pad_idx] * (seq_length - len(input_ids)) + input_ids
    else:
        input_ids = input_ids[-seq_length:]

    generated = []
    for _ in range(num_generate):
        x = np.array([input_ids], dtype=np.int32)  # shape (1, seq_length)
        preds = model.predict(x, verbose=0)        # shape (1, vocab_size)
        preds = preds[0].astype(np.float64)

        # Temperature scaling and sampling
        preds = np.log(preds + 1e-8) / max(1e-8, temperature)
        exp_preds = np.exp(preds)
        probs = exp_preds / np.sum(exp_preds)

        next_id = np.random.choice(range(vocab_size), p=probs)
        next_char = idx2char[next_id]
        generated.append(next_char)

        input_ids = input_ids[1:] + [next_id]
    return seed_text + ''.join(generated)

In [11]:
SEED = "MENENIUS:"
NUM_GENERATE = 1000
TEMPERATURE = 0.7

sample = generate_text(SEED, NUM_GENERATE, TEMPERATURE)
print("\n--- Generated sample ---\n")
print(sample)
print("\n--- End sample ---\n")


--- Generated sample ---

MENENIUS::
Oiolis poy cole botee
Wf boa woe she aatot yoe soaoe soics eo prseus wols toru, I I oo thhar tuan Bosior inle aoras yome macirs Ios wes tog lor ther,
Yhih wrd mooee nne surle oonree theut,

nn foars yoeog liln tor biy gomy a soe wisiwod tor bin uotrs as or hert soat the thoow, I
I tan soue I aoe ghith nig sale, aaf whis snue
the iithe simiee tor ionrrt wato a s ane iat yris sat, oins fan getir ao the ters aissren
nos loo onre thhale loes
Bat, thto loe toin ae an ane mhe cos hhy tan o cnre so w sae sosrr
h lorr pht srer, Iow thosee inrin oors ghesre toraln nousle iarns drs tnis tone to ngd an coot, ae soad car nse thaee borl thes aoot sr arnn.

SCSONO:
IN,
INI
Iu:
Iy mel oics toe moe in ges tote I yhs the done gho itee casn eateaa witison tim toao tile len an tirat ourrre thas son aaaes at sore
hhe sla biss thu snr, los yriod bistrr
yo r motte Moun toetn waed tor hor tow lead thas thotel ad tilime
In oot ant heihhe tiy thlee
To tos theers ar gon the